In [1]:
import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
import torch
import torch.nn as nn

class CrossNetwork(nn.Module):
    def __init__(self, input_dim, num_layers):
        super(CrossNetwork, self).__init__()
        self.num_layers = num_layers
        # Trọng số và bias cho mỗi lớp cross
        self.cross_weights = nn.ParameterList([nn.Parameter(torch.randn(input_dim)) for _ in range(num_layers)])
        self.cross_bias = nn.ParameterList([nn.Parameter(torch.zeros(input_dim)) for _ in range(num_layers)])

    def forward(self, x0):
        x_l = x0
        for i in range(self.num_layers):
            # Tính (x_l^T * w_l) -> scalar cho mỗi sample trong batch
            xl_w = torch.matmul(x_l, self.cross_weights[i])
            xl_w = xl_w.unsqueeze(1) 
            # Cập nhật: x_{l+1} = x_0 * (x_l^T * w_l) + b_l + x_l
            x_l = x0 * xl_w + self.cross_bias[i] + x_l
        return x_l

class DCN(nn.Module):
    def __init__(self, num_dense_features, sparse_vocab_sizes, embed_dim, cross_layers, hidden_dims):
        super(DCN, self).__init__()
        # Embedding cho sparse features
        self.embeddings = nn.ModuleList([nn.Embedding(vocab, embed_dim) for vocab in sparse_vocab_sizes])
        
        input_dim = num_dense_features + len(sparse_vocab_sizes) * embed_dim
        
        # Mạng Cross
        self.cross_net = CrossNetwork(input_dim, cross_layers)
        
        # Mạng Deep
        deep_layers = []
        in_dim = input_dim
        for dim in hidden_dims:
            deep_layers.append(nn.Linear(in_dim, dim))
            deep_layers.append(nn.ReLU())
            in_dim = dim
        self.deep_net = nn.Sequential(*deep_layers)
        
        # Lớp Combination để dự đoán cuối cùng (Output Layer)
        self.fc = nn.Linear(input_dim + in_dim, 1)

    def forward(self, dense_x, sparse_x):
        # Embedding các giá trị sparse
        emb_x = [emb(sparse_x[:, i]) for i, emb in enumerate(self.embeddings)]
        emb_x = torch.cat(emb_x, dim=1)
        
        # Nối (stack) dense features và embedded sparse features
        x0 = torch.cat([dense_x, emb_x], dim=1)
        
        # Đi qua 2 mạng song song
        cross_out = self.cross_net(x0)
        deep_out = self.deep_net(x0)
        
        # Nối đầu ra
        combined = torch.cat([cross_out, deep_out], dim=1)
        
        # Đi qua lớp Linear cuối cùng để lấy logits
        out = self.fc(combined)
        
        # Ép về 1D (bỏ sigmoid) để tương thích với BCEWithLogitsLoss
        return out.squeeze(1)

In [5]:
import math

class StandardCriteoDataset(Dataset):
    def __init__(self, hf_dataset, dense_cols, cat_cols):
        self.data = hf_dataset
        self.dense_cols = dense_cols
        self.cat_cols = cat_cols
        
        self.block_vocab_sizes = []
        self.cat_mappers = []
        
        print("Đang đếm Vocabulary Size cho 26 khối Categorical...")
        self._build_vocab()
        
    def _build_vocab(self):
        df = self.data.to_pandas()
        for col in tqdm(self.cat_cols, desc="Building Vocab"):
            unique_vals = df[col].dropna().unique()
            mapper = {val: idx + 1 for idx, val in enumerate(unique_vals)}
            self.cat_mappers.append(mapper)
            self.block_vocab_sizes.append(len(unique_vals) + 1)
            
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        
        dense_x = []
        for c in self.dense_cols:
            val = row[c]
            # Nếu giá trị là None hoặc số âm, gán bằng 0
            if val is None or val < 0:
                dense_x.append(0.0)
            else:
                # Công thức chuẩn hóa: log(1 + giá_trị)
                dense_x.append(math.log1p(float(val))) 
        
        # Categorical
        cat_ids = []
        for col_idx, col_name in enumerate(self.cat_cols):
            val = row[col_name]
            mapper = self.cat_mappers[col_idx]
            cat_ids.append(mapper.get(val, 0)) 
            
        label = float(row['label'])

        return (
            torch.tensor(dense_x, dtype=torch.float32),
            torch.tensor(cat_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32)
        )

In [6]:
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, random_split
import glob
import os

# =========================
# 1. Khai báo cột
# =========================
dense_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

# =========================
# 2. Load parquet local
# =========================
print("Đang tải dữ liệu parquet local...")

# đường dẫn folder chứa các file parquet
data_dir = "/kaggle/input/datasets/huy291/criteo-cleaned-data/data"

# lấy toàn bộ parquet
parquet_files = sorted(glob.glob(os.path.join(data_dir, "*.parquet")))

print(f"Số file parquet: {len(parquet_files)}")

# load bằng HuggingFace datasets
raw_dataset = load_dataset(
    "parquet",
    data_files=parquet_files,
    split="train"
)

print(raw_dataset)

# =========================
# 3. Dataset wrapper
# =========================
dataset = StandardCriteoDataset(
    raw_dataset,
    dense_cols,
    cat_cols
)

# =========================
# 4. Chia train/val
# =========================
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

print(f"Tổng số mẫu: {len(dataset)}")
print(f"Train: {train_size}")
print(f"Val: {val_size}")

# =========================
# 5. DataLoader
# =========================
train_loader = DataLoader(
    train_dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Hoàn tất tạo DataLoader.")

Đang tải dữ liệu parquet local...
Số file parquet: 50


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/34 [00:00<?, ?it/s]

Dataset({
    features: ['label', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'I10', 'I11', 'I12', 'I13', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26'],
    num_rows: 45840617
})
Đang đếm Vocabulary Size cho 26 khối Categorical...


Building Vocab:   0%|          | 0/26 [00:00<?, ?it/s]

Tổng số mẫu: 45840617
Train: 41256555
Val: 4584062
Hoàn tất tạo DataLoader.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

model = DCN(
    num_dense_features=len(dense_cols),
    sparse_vocab_sizes=dataset.block_vocab_sizes,
    embed_dim=64,                      # Số chiều embedding cố định cho mọi cột
    cross_layers=3,                    # Số lớp trong Cross Network
    hidden_dims=[256, 128, 64]         # Cấu trúc của Deep Network
)

# Wrap DataParallel nếu có >= 2 GPU
if num_gpus > 1:
    print(f"Dùng DataParallel trên {num_gpus} GPU")
    model = nn.DataParallel(model)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

print("Bắt đầu huấn luyện...")
EPOCHS = 3

for epoch in range(EPOCHS):
    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0
    train_targets = []
    train_preds = []

    train_bar = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{EPOCHS}")

    for dense_x, cat_x, labels in train_bar:
        # pin_memory + non_blocking giúp transfer CPU→GPU nhanh hơn
        dense_x = dense_x.to(device, non_blocking=True)
        cat_x   = cat_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(dense_x, cat_x)

        # DataParallel trả về output ghép từ nhiều GPU → shape vẫn đúng
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        # .mean() để lấy scalar khi DataParallel trả về tensor nhiều phần tử
        train_loss += loss.item()

        probs = torch.sigmoid(logits)
        train_targets.extend(labels.detach().cpu().numpy())
        train_preds.extend(probs.detach().cpu().numpy())

        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)
    train_auc = roc_auc_score(train_targets, train_preds)

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0
    val_targets = []
    val_preds = []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}/{EPOCHS}", leave=False)
        for dense_x, cat_x, labels in val_bar:
            dense_x = dense_x.to(device, non_blocking=True)
            cat_x   = cat_x.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)

            logits = model(dense_x, cat_x)
            loss   = criterion(logits, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            val_targets.extend(labels.cpu().numpy())
            val_preds.extend(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_auc = roc_auc_score(val_targets, val_preds)

    print(
        f"Epoch {epoch+1}: \n"
        f"Train Loss: {avg_train_loss:.4f} | Train AUC: {train_auc:.4f} || "
        f"Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}\n"
    )

Số GPU khả dụng: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition
Bắt đầu huấn luyện...


[TRAIN] Epoch 1/3: 100%|██████████| 10073/10073 [22:51<00:00,  7.34it/s, Loss=0.4668] 


Epoch 1: Train Loss: 0.6623 | Train AUC: 0.7762 || Val Loss: 0.4604 | Val AUC: 0.7898



[TRAIN] Epoch 2/3: 100%|██████████| 10073/10073 [22:53<00:00,  7.33it/s, Loss=0.3825]


Epoch 2: Train Loss: 0.4060 | Train AUC: 0.8420 || Val Loss: 0.5051 | Val AUC: 0.7631



[TRAIN] Epoch 3/3: 100%|██████████| 10073/10073 [22:58<00:00,  7.31it/s, Loss=0.3748]


Epoch 3: Train Loss: 0.3814 | Train AUC: 0.8614 || Val Loss: 0.5303 | Val AUC: 0.7509

